## 1. Load

In [ ]:
# SETUP
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/clean_full.csv", parse_dates=["order_date_dateorders"])
print("Shape:", df.shape)
print("Orders:", df["order_id"].nunique())
print("First order:", df["order_date_dateorders"].min())
print("Last order:", df["order_date_dateorders"].max())

Shape: (172765, 22)
Orders: 62897
First order: 2015-01-01 00:00:00
Last order: 2018-01-31 23:38:00


## 2. One row per order

Can the item rows of one order become one row? Only if the order-level columns have one value per order. Every line below must say 1.

In [3]:
order_cols = ["order_date_dateorders", "late_delivery_risk", "type", "days_for_shipment_scheduled",
              "shipping_mode", "market", "order_region", "order_country", "order_state",
              "order_city", "customer_segment"]
print(df.groupby("order_id")[order_cols].nunique().max())

order_date_dateorders          1
late_delivery_risk             1
type                           1
days_for_shipment_scheduled    1
shipping_mode                  1
market                         1
order_region                   1
order_country                  1
order_state                    1
order_city                     1
customer_segment               1
dtype: int64


In [4]:
keep = ["order_id", "order_date_dateorders", "late_delivery_risk", "type", "shipping_mode",
        "market", "order_region", "order_country", "order_state", "order_city", "customer_segment"]
sums = df.groupby("order_id").agg(order_total_sales=("sales", "sum"), items_in_order=("sales", "size"))
orders = df[keep].drop_duplicates("order_id").join(sums, on="order_id").reset_index(drop=True)
print("Orders shape:", orders.shape)
print("Item rows covered:", orders["items_in_order"].sum())

Orders shape: (62897, 13)
Item rows covered: 172765


## 3. Split by date

In [5]:
orders = orders.sort_values(["order_date_dateorders", "order_id"]).reset_index(drop=True)
dates = orders["order_date_dateorders"]
cuts = []
for cut in (round(len(orders) * 0.70), round(len(orders) * 0.85)):
    while dates[cut] == dates[cut - 1]:   # same minute on both sides, so move the cut
        cut += 1
    cuts.append(cut)
orders["split"] = "test"
orders.loc[orders.index < cuts[0], "split"] = "train"
orders.loc[(orders.index >= cuts[0]) & (orders.index < cuts[1]), "split"] = "val"
print("Cut positions:", cuts)

Cut positions: [44028, 53462]


In [6]:
summary = orders.groupby("split").agg(
    orders=("order_id", "size"),
    first_order=("order_date_dateorders", "min"),
    last_order=("order_date_dateorders", "max"),
    late_share=("late_delivery_risk", "mean"),
).loc[["train", "val", "test"]].round(4)
summary

,orders,first_order,last_order,late_share
split,,,,
train,44028,2015-01-01 00:00:00,2017-03-16 15:09:00,0.5733
val,9434,2017-03-16 15:51:00,2017-09-05 13:20:00,0.5702
test,9435,2017-09-05 13:41:00,2018-01-31 23:38:00,0.5754


In [7]:
print("Order ids unique:", orders["order_id"].is_unique)
print("Train ends before val starts:", summary.loc["train", "last_order"] < summary.loc["val", "first_order"])
print("Val ends before test starts:", summary.loc["val", "last_order"] < summary.loc["test", "first_order"])
print("Counts add up:", summary["orders"].sum() == df["order_id"].nunique())

Order ids unique: True
Train ends before val starts: True
Val ends before test starts: True
Counts add up: True


In [8]:
is_train, is_val, is_test = (orders["split"] == s for s in ["train", "val", "test"])
print(is_train.sum(), is_val.sum(), is_test.sum())

44028 9434 9435


## 4. Time features
Extract hour, day, month, weekend and holiday-season features from the order date.

In [9]:
d = orders["order_date_dateorders"]
orders["order_hour"] = d.dt.hour
orders["order_day_of_week"] = d.dt.dayofweek   # 0 = Monday
orders["order_month"] = d.dt.month
orders["is_weekend"] = (orders["order_day_of_week"] >= 5).astype(int)
orders["order_is_holiday_season"] = d.dt.month.isin([11, 12]).astype(int)
orders["hour_bucket"] = pd.cut(orders["order_hour"], bins=[-1, 5, 11, 17, 23],
                               labels=["night", "morning", "afternoon", "evening"]).astype(str)

In [10]:
time_cols = ["order_hour", "order_day_of_week", "order_month", "is_weekend", "order_is_holiday_season"]
print(orders[time_cols].agg(["min", "max"]))
print(orders["hour_bucket"].value_counts())

     order_hour  order_day_of_week  order_month  is_weekend  \
min           0                  0            1           0   
max          23                  6           12           1   

     order_is_holiday_season  
min                        0  
max                        1  
hour_bucket
evening      15783
night        15758
morning      15704
afternoon    15652
Name: count, dtype: int64


Hour runs from 0 to 23, day of week from 0 to 6 and month from 1 to 12. Both flags are 0 or 1. The four hour buckets hold about a quarter of the orders each, from 15,652 to 15,783.

## 5. Same Day cutoff

Identify whether a Same Day order was placed at or after the cutoff hour selected using training data.

In [11]:
same_day = orders["shipping_mode"] == "Same Day"
rate_by_hour = orders[is_train & same_day].groupby("order_hour")["late_delivery_risk"].mean()
cutoff = rate_by_hour[rate_by_hour > 0.5].index.min()
print("Same Day cutoff hour (from train):", cutoff)

Same Day cutoff hour (from train): 12


In [12]:
orders["same_day_after_cutoff"] = (same_day & (orders["order_hour"] >= cutoff)).astype(int)
check = orders[(is_train | is_val) & same_day]
check.groupby(["split", "same_day_after_cutoff"])["late_delivery_risk"].agg(orders="size", late_rate="mean").round(3)

orders  late_rate
split same_day_after_cutoff                   
train 0                        1271        0.0
      1                        1100        1.0
val   0                         250        0.0
      1                         278        1.0

The cutoff learned from train is hour 12. In train, Same Day orders before it have a late rate of 0.0 (1,271 orders), and from hour 12 on it is 1.0 (1,100 orders). Validation shows the same: 0.0 (250 orders) and 1.0 (278 orders). The pattern holds outside train.

## 6. Location Features

Encode country, state and city using order shares calculated from training data.



In [13]:
place_cols = ["order_country", "order_state", "order_city"]
for col in place_cols:
    share = orders.loc[is_train, col].value_counts(normalize=True)   # share of train orders
    orders[col + "_freq"] = orders[col].map(share).fillna(0)          # unseen value gets 0
print(orders.loc[is_train, [c + "_freq" for c in place_cols]].agg(["min", "max"]))

     order_country_freq  order_state_freq  order_city_freq
min            0.000023          0.000023         0.000023
max            0.179136          0.035546         0.016217


In [14]:
rows = []
for col in place_cols:
    seen = set(orders.loc[is_train, col])
    rows.append({"column": col, "values_in_train": len(seen),
                 "val_unseen": (~orders.loc[is_val, col].isin(seen)).sum(),
                 "test_unseen": (~orders.loc[is_test, col].isin(seen)).sum()})
pd.DataFrame(rows)

,column,values_in_train,val_unseen,test_unseen
0,order_country,164,0,0
1,order_state,1080,2,1
2,order_city,3528,42,19


## 7. Order size 

Create order-size features and check whether order sizes change across the train, validation and test periods.

In [15]:
high_value_cutoff = orders.loc[is_train, "order_total_sales"].quantile(0.75)
orders["is_high_value"] = (orders["order_total_sales"] > high_value_cutoff).astype(int)
print("High-value cutoff (train 75th percentile):", round(high_value_cutoff, 2))

High-value cutoff (train 75th percentile): 829.92


In [16]:
size_cols = ["items_in_order", "order_total_sales", "is_high_value"]
orders.groupby("split")[size_cols].mean().loc[["train", "val", "test"]].round(3)

,items_in_order,order_total_sales,is_high_value
split,,,
train,3.002,591.897,0.249
val,2.990,614.225,0.273
test,1.312,356.097,0.098


The high-value cutoff learned from train is 829.92. Orders get much smaller in the test period: 1.31 items and an average total of 356.10, against 3.00 items and 591.90 in train. Only about 10% of test orders are above the train cutoff, against about 25% in train and 27% in validation. These columns describe a different kind of order in the last period (see the EDA notebook), so they are candidates only.

## 8. Combined columns

Test combinations of shipping mode with time, location and order-size features.

In [17]:
orders["mode_hour"] = orders["shipping_mode"] + "_" + orders["hour_bucket"]
orders["mode_weekday"] = orders["shipping_mode"] + "_" + orders["order_day_of_week"].astype(str)
orders["mode_region"] = orders["shipping_mode"] + "_" + orders["order_region"]
size_bucket = orders["items_in_order"].clip(upper=3).map({1: "1", 2: "2", 3: "3_or_more"})
orders["mode_size"] = orders["shipping_mode"] + "_" + size_bucket

In [18]:
hour = orders["order_hour"]
period = pd.Series("not_same_day", index=orders.index)
period.loc[same_day & (hour < cutoff)] = "same_day_morning"
period.loc[same_day & (hour >= cutoff) & (hour < 18)] = "same_day_afternoon"
period.loc[same_day & (hour >= 18)] = "same_day_evening"
orders["same_day_period"] = period

In [19]:
combined_cols = ["mode_hour", "mode_weekday", "mode_region", "mode_size", "same_day_period"]
rows = []
for col in combined_cols:
    seen = set(orders.loc[is_train, col])
    rows.append({"column": col, "labels": orders[col].nunique(), "in_train": len(seen),
                 "val_unseen": (~orders.loc[is_val, col].isin(seen)).sum(),
                 "test_unseen": (~orders.loc[is_test, col].isin(seen)).sum()})
pd.DataFrame(rows)

,column,labels,in_train,val_unseen,test_unseen
0,mode_hour,16,16,0,0
1,mode_weekday,28,28,0,0
2,mode_region,92,92,0,0
3,mode_size,12,12,0,0
4,same_day_period,4,4,0,0


In [20]:
orders[is_train].groupby("same_day_period")["late_delivery_risk"].agg(orders="size", late_rate="mean").round(3)

,orders,late_rate
same_day_period,,
not_same_day,41657,0.579
same_day_afternoon,561,1.000
same_day_evening,539,1.000
same_day_morning,1271,0.000


## 9. Check the tables


In [21]:
KEYS = ["order_id", "order_date_dateorders", "late_delivery_risk"]
text_cols = ["type", "shipping_mode", "market", "order_region", "customer_segment", "hour_bucket",
             "mode_hour", "mode_weekday", "mode_region", "mode_size", "same_day_period"]
numeric_cols = ["order_hour", "order_day_of_week", "order_month", "is_weekend", "order_is_holiday_season",
                "same_day_after_cutoff", "order_country_freq", "order_state_freq", "order_city_freq",
                "order_total_sales", "items_in_order", "is_high_value"]
all_cols = KEYS + text_cols + numeric_cols
train_features = orders.loc[is_train, all_cols]
val_features = orders.loc[is_val, all_cols]
test_features = orders.loc[is_test, all_cols]

In [22]:
tables = {"train": train_features, "val": val_features, "test": test_features}
print("Same columns in all three:", list(train_features.columns) == list(val_features.columns) == list(test_features.columns))
print("Missing values:", sum(t.isnull().sum().sum() for t in tables.values()))
print("Shapes:", {name: t.shape for name, t in tables.items()})
print("Non-numeric among the numeric columns:", train_features[numeric_cols].select_dtypes(exclude="number").shape[1])

Same columns in all three: True
Missing values: 0
Shapes: {'train': (44028, 26), 'val': (9434, 26), 'test': (9435, 26)}
Non-numeric among the numeric columns: 0


In [23]:
for col in ["type", "shipping_mode", "market", "customer_segment", "hour_bucket", "same_day_period"]:
    print(col, sorted(train_features[col].unique()))

type ['CASH', 'DEBIT', 'PAYMENT', 'TRANSFER']
shipping_mode ['First Class', 'Same Day', 'Second Class', 'Standard Class']
market ['Africa', 'Europe', 'LATAM', 'Pacific Asia', 'USCA']
customer_segment ['Consumer', 'Corporate', 'Home Office']
hour_bucket ['afternoon', 'evening', 'morning', 'night']
same_day_period ['not_same_day', 'same_day_afternoon', 'same_day_evening', 'same_day_morning']


## 10. Save


In [24]:
for name, t in tables.items():
    t.to_csv(f"../data/processed/{name}_features.csv", index=False)

In [25]:
for name, t in tables.items():
    back = pd.read_csv(f"../data/processed/{name}_features.csv", parse_dates=["order_date_dateorders"])
    print(name, "| same shape:", back.shape == t.shape, "| same columns:", list(back.columns) == list(t.columns),
          "| missing:", back.isnull().sum().sum(), "| date type:", back["order_date_dateorders"].dtype)

train | same shape: True | same columns: True | missing: 0 | date type: datetime64[us]
val | same shape: True | same columns: True | missing: 0 | date type: datetime64[us]
test | same shape: True | same columns: True | missing: 0 | date type: datetime64[us]
